# Point-in-time vertical slice

This notebook queries normalized daily prices, SEC fundamentals, and FRED macro Parquet in one DuckDB session. Joins use publication timestamps: a row is visible only after it was publicly knowable. No sample observations are fabricated. On a fresh checkout the catalog exposes typed empty views and this notebook explains what ingestion must run first.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from research import ResearchCatalog

candidate = Path(os.getenv("INVS_DATA_ROOT", "/data"))
DATA_ROOT = candidate if candidate.exists() else Path.cwd().parent.parent / "data"
catalog = ResearchCatalog(DATA_ROOT).register()
pd.DataFrame([item.__dict__ for item in catalog.status()])

## Select a reproducible slice

Environment variables can pin the price security, its mapped SEC issuer, SEC concept, and FRED series. Otherwise, the lexicographically first available value is selected so reruns against the same dataset are deterministic. For a multi-security universe, set `EXAMPLE_SECURITY_ID` and `EXAMPLE_ISSUER_ID` to a pair from the configured security master.

In [ ]:
def configured_or_first(variable, query):
    configured = os.getenv(variable)
    if configured:
        return configured
    row = catalog.connection.execute(query).fetchone()
    return row[0] if row else None

security_id = configured_or_first(
    "EXAMPLE_SECURITY_ID",
    "SELECT security_id FROM prices WHERE security_id IS NOT NULL ORDER BY security_id LIMIT 1",
)
issuer_id = configured_or_first(
    "EXAMPLE_ISSUER_ID",
    "SELECT issuer_id FROM fundamentals WHERE issuer_id IS NOT NULL ORDER BY issuer_id LIMIT 1",
)
fundamental_concept = configured_or_first(
    "EXAMPLE_SEC_CONCEPT",
    "SELECT concept FROM fundamentals WHERE concept IS NOT NULL ORDER BY concept LIMIT 1",
)
macro_series_id = configured_or_first(
    "EXAMPLE_FRED_SERIES",
    "SELECT series_id FROM macroeconomics WHERE series_id IS NOT NULL ORDER BY series_id LIMIT 1",
)
selection = {
    "security_id": security_id,
    "issuer_id": issuer_id,
    "fundamental_concept": fundamental_concept,
    "macro_series_id": macro_series_id,
}
selection

In [ ]:
if any(value is None for value in selection.values()):
    missing = ", ".join(catalog.missing()) or "one or more populated datasets"
    print(
        "No joined observations yet. Run the price, SEC, and FRED collectors first. "
        f"Missing or empty inputs: {missing}. The empty result below is expected on first boot."
    )
    joined = pd.DataFrame()
else:
    joined = catalog.point_in_time_frame(
        security_id=security_id,
        issuer_id=issuer_id,
        fundamental_concept=fundamental_concept,
        macro_series_id=macro_series_id,
    )

joined.tail(20)

## Lookahead guard

These assertions document the temporal contract. They tolerate missing fundamentals or macro values, but any joined value must have been published no later than the price row's `known_at` timestamp.

In [ ]:
if not joined.empty:
    known_at = pd.to_datetime(joined["known_at"], utc=True)
    fundamental_available = pd.to_datetime(joined["fundamental_available_at"], utc=True)
    macro_available = pd.to_datetime(joined["macro_available_at"], utc=True)
    assert (fundamental_available.dropna() <= known_at[fundamental_available.notna()]).all()
    assert (macro_available.dropna() <= known_at[macro_available.notna()]).all()
    print(f"Validated {len(joined):,} price rows without publication-time lookahead.")
else:
    print("Lookahead checks skipped because normalized inputs have not been ingested yet.")